# **Autograd Engine**

# Micrograd(Scalar)

In [ ]:
import os

dir_path = "custom_autograd"

# Checks if directory dir_path already exists, then print "already exists", else create directory
if os.path.isdir(dir_path):
  print(f"{dir_path} already exists")
else:
  os.makedirs("custom_autograd")

In [ ]:
%%writefile custom_autograd/engine.py

import math

_TRACK_GRADIENTS = True # flag for gradient tracking

# Class no_grad to switch off gradient tracking during inference
class no_grad:

  def __enter__(self):
    global _TRACK_GRADIENTS
    self.prev = _TRACK_GRADIENTS
    _TRACK_GRADIENTS = False


  def __exit__(self, *args):
    global _TRACK_GRADIENTS
    _TRACK_GRADIENTS = self.prev


# Class Value to handle main engine calculus and backpropagation
class Value:

  def __init__(self, data, _children=(), _op=""):
    self.data = float(data) # stores data

    self.grad = 0.0 # stores gradient

    self._prev = set(_children) # set of parent nodes that created current node

    self._op = _op if _TRACK_GRADIENTS else ""# operation used to create current node

    self._backward = lambda: None # buffer backward function that will be replaced

    # static replay pointer cache
    self._compiled_topology = None
    self._forward_replay = lambda: None


  @staticmethod
  def _autowrap(other):
    # autowraps scalar value to Value class object
    return other if isinstance(other, Value) else Value(other)


  def _build_topo(self):
    # builds custom graph topology for backpropagation
    topo = []
    visited = set()

    def dfs(node):
      if node not in visited:
        visited.add(node)
        for parent in node._prev:
          dfs(parent)
        topo.append(node)
    dfs(self)
    return topo


  def compile_topology(self):
    # using _build_topo function
    return self._build_topo()


  def __add__(self, other):
    # 1. constanct fallback(Autowrapping)
    # check if other is a Value object
    other = self._autowrap(other)

    # 2. forward Pass:
    out = Value(self.data+other.data, _children=(self, other), _op="+")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * 1.0
      other.grad += out.grad * 1.0

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = self.data + other.data
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def __radd__(self, other):
    # reverse addition
    return self + other


  def __mul__(self, other):
    # 1. autowrapping
    other = self._autowrap(other)

    # 2. forward pass
    out = Value(self.data*other.data, _children=(self, other), _op="*")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * other.data
      other.grad += out.grad * self.data

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = self.data * other.data
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def __rmul__(self, other):
    # reverse multiplication
    return self * other


  def __pow__(self, other):
    # checking if other is an integer or float value
    assert isinstance(other, (int, float))

    # 2. forward pass
    out = Value(self.data**other, _children=(self,), _op="**")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * (other*self.data**(other-1))

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = self.data**other
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def __rpow__(self, other):
    # reverse power function
    return self._autowrap(other)**self


  def __sub__(self, other):
    # 1. autowrapping
    other = self._autowrap(other)

    # 2. forward pass
    out = Value(self.data-other.data, _children=(self, other), _op="-")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * 1.0
      other.grad += out.grad * (-1.0)

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = self.data - other.data
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def __rsub__(self, other):
    # reverse subtraction
    return self._autowrap(other) - self


  def __truediv__(self, other):
    # 1. autowrapping
    other = self._autowrap(other)

    # 2. forward pass
    out = Value(self.data/other.data, _children=(self, other), _op="/")

    # 3. local backward pass
    def _backward():
      self.grad += out.grad * 1/other.data
      other.grad += out.grad * (self.data*-1.0*(other.data**-2))

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = self.data / other.data
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def __rtruediv__(self, other):
    # reverse division
    return self._autowrap(other) / self


  def __neg__(self):
    # 2. forward pass
    out = Value(-self.data, _children=(self,), _op="neg")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * -1.0

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = -self.data
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def __rneg__(self):
    # reverse negation
    return -self


  def backward(self, compiled_path=None):
    # using compiled path for backpropagation
    if compiled_path is not None:
      self.grad = 1.0
      for node in reversed(compiled_path):
        node._backward()
      return

    # fallback to full dynamic sorting if not pre compiled
    # 1. custom DFS Graph sort
    topo = self._build_topo()

    # 2. set the initial output gradient to 1.0
    self.grad = 1.0

    # 3. walk the sorted graph in reverse
    for node in reversed(topo):
      node._backward()


  def tanh(self):
    # 2. forward pass
    next_h = math.tanh(self.data)
    out = Value(next_h, _children=(self,), _op="tanh")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * (1-out.data*out.data)

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = math.tanh(self.data)
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward

    return out # returning the resulting Value object


  def relu(self):
    # 2. forward pass
    x = max(0, self.data)
    out = Value(x, _children=(self,), _op="relu")

    # 3. local backward function
    def _backward():
      self.grad += out.grad * (1.0 if out.data>0.0 else 0.0)

    # 4. local forward replay function for backpropagation using static cache
    def _forward_replay():
      out.data = max(0.0, self.data)
    out._forward_replay = _forward_replay

    # 5. toogling gradient tracking for backpropagation
    if _TRACK_GRADIENTS:
      out._backward = _backward
    return out


  def __repr__(self):
    # function for custom representation used for debugging.
    return f"Value(data={self.data}, grad={self.grad})"


# Class FastCompiledTopology for static sped up engine nodes topology
class FastCompiledTopology:

  def __init__(self, loss_node, input_placeholders, target_placeholders):
    self.loss_node = loss_node # loss node

    self.input_placeholders = input_placeholders # input placeholders or data values
    self.target_placeholders = target_placeholders # target placeholders or target values

    self.topo_order = loss_node._build_topo() # stores the topology order using _build_topo to build static topology

    self.backward_order = [node for node in self.topo_order if len(node._prev)>0] # stores only nodes that has parents, implements dead end pruning


  def update_batch(self, batch_x, batch_y):
    # in place updates for training and targets
    for placeholder_row, sample_x in zip(self.input_placeholders, batch_x):
      for node, val_x in zip(placeholder_row, sample_x):
        node.data = float(val_x)

    for placeholder_node, val_y in zip(self.target_placeholders, batch_y):
      placeholder_node.data = float(val_y)


  def forward(self):
    # replaying the forward pass
    for node in self.topo_order:
      node._forward_replay()
    return self.loss_node.data


  def backward(self, parameters):
    # performing/replaying backward pass
    for node in self.topo_order:
      node.grad = 0.0
    for p in parameters:
      p.grad = 0.0

    self.loss_node.grad = 1.0
    for node in reversed(self.backward_order):
      node._backward()


  def predict(self, val_x):
    # only for inference purposes
    for placeholders_row, sample_x in zip(self.input_placeholders, val_x):
      for node, val in zip(placeholders_row, sample_x):
        node.data = float(val)

    # forward pass
    for node in self.topo_order:
      node._forward_replay()

    return self.loss_node.data # returns loss

In [ ]:
%%writefile custom_autograd/nn.py

import random
from custom_autograd.engine import Value

class Neuron:

  def __init__(self, nin):
    # initializing the weights
    self.weights = [Value(random.uniform(-1, 1)) for _ in range(nin)]

    # initializing the bias
    self.bias = Value(random.uniform(-1, 1))


  def __call__(self, x):
    # multiplying every input by it's corresponding weight
    bi = self.bias
    for wi, xi in zip(self.weights, x):
      bi = bi + wi * xi
    return bi.tanh()


  def parameters(self):
    # returning model parameters
    return self.weights + [self.bias]


class Layer:

  def __init__(self, nin, nout):
    # Create a list of 'nout' distinct neurons
    self.neurons = [Neuron(nin) for _ in range(nout)]


  def __call__(self, x):
    # Passing input list x to every single neuron and making a list of their outputs
    outputs = [neuron(x) for neuron in self.neurons]

    # single neuron - return salar value object, multi neuron - return output list
    return outputs


  def parameters(self):
    # store weights and bais for each neuron
    params = []
    for neuron in self.neurons:
      params.extend(neuron.parameters())
    return params


class MLP:

  def __init__(self, nin, nouts):
    # 1. Combine input dimensions with output dimensions
    sizes = [nin] + nouts

    # 2. Build the layers sequentially by mapping adjacent sizes
    self.layers = []
    for i in range(len(nouts)):
      self.layers.append(Layer(sizes[i], sizes[i+1]))


  def __call__(self, x):
    # Feeds the input vector sequentially through every single layer
    for layer in self.layers:
      x = layer(x)
    return x[0]


  def parameters(self):
    params = []
    for layer in self.layers:
      params.extend(layer.parameters())
    return params

In [ ]:
%%writefile custom_autograd/test.py

# testing the scaler autograd(basic operations testing)
a = Value(2.0)
b = Value(5.0)

c = a + b
print(f"c={c}")
print(f"parents={c._prev}")

d = a + 2.0
print(f"d={d}")

e = 10.0 + b
print(f"e={e}")

print("Simulating the backward pass")
c.grad = 1.0
c.backward()

print(f"Gradient of a={a.grad}")
print(f"fGradient of b={b.grad}")

In [ ]:
%%writefile custom_autograd/data.py

import random
import numpy as np

# Generating synthetic Binary Classification data and splits the data into training and testing datasets

def generate_data(num_samples: int=100, num_features: int=3, low: float=-2.0, high: float=2.0, random_seed: int=None):
  '''
    generates synthetic binary classification data for the model
    Args:
      num_samples = number of samples(default: 100),
      num_features = number of features(default: 3),
      low = float value representing lower bound for data(default: -2.0)
      high = float value representing higher bound for data(default: 2.0)
      random_seed = integer value to set the random seed(default: None)

    Example:
      generate_data(num_samples=1000, num_features=5, low=-2.0, high=2.0, random_seed=42)
  '''
  if random_seed:
    random.seed(42)

  X = []
  y = []
  for sample in range(num_samples):
    row = [random.uniform(low, high) for _ in range(num_features)]
    X.append(row)

    if sum(row) > 0.0:
      y.append(1)
    else:
      y.append(-1)

  return X, y


def split_data(data: list=[], labels: list= [], split_value: float=0.8):
  '''
    Splits the data into training and testing datasets and returns them
    Args:
      data = list containing data
      split_value = float value to split data(0.0 to 1.0)

    Example:
      split_data(data=data_list, split_value=0.8)
  '''
  if split_value < 0.0 or split_value > 1.0:
    raise Exception("split_value must be between 0.0 and 1.0")
    return

  n = int(split_value*len(data))
  train_data, train_labels = data[:n], labels[:n]
  test_data, test_labels = data[n:], labels[n:]

  return train_data, train_labels, test_data, test_labels


def generate_xor_checkerboard(num_samples: int=100, num_features: int=2, low: float=-2.0, high: float=2.0, noise: float=0.1, random_seed: int=None):
  '''
    generated xor checkeboard pattern synthetic data for model training
    Args:
      num_samples = number of samples(default: 100),
      num_features = number of features(default: 3),
      low = float value representing lower bound for data(default: -2.0)
      high = float value representing higher bound for data(default: 2.0)
      random_seed = integer value to set the random seed(default: None)

    Example:
      generate_xor_checkerboard(num_samples=1000, num_features=5, low=-2.0, high=2.0, random_seed=42)
  '''

  if random_seed:
    random.seed(42)

  X = []
  y = []
  for sample in range(num_samples):
    X = np.random.uniform(low, high, (num_samples, num_features))
    y = np.sign(X[:, 0]*X[:, 1])

    X += np.random.normal(0, noise, X.shape)

  return X.tolist(), y.tolist()

In [ ]:
%%writefile custom_autograd/train.py

import argparse
import numpy as np
import matplotlib.pyplot as plt
from custom_autograd.engine import Value, no_grad, FastCompiledTopology
from custom_autograd.nn import MLP
from custom_autograd.data import generate_data, split_data
from tqdm.auto import tqdm

# Plotting and visualizing the results
def plot_results(model_name, epochs: list, train_loss: list, test_loss: list, train_accuracy: list, test_accuracy: list):

  # 1. data sanitization
  unp_epochs = [int(e) for e in epochs]
  unp_train_loss = [float(tl.data) if hasattr(tl, 'data') else float(tl) for tl in train_loss]
  unp_test_loss = [float(vl.data) if hasattr(vl, 'data') else float(vl) for vl in test_loss]
  unp_train_acc = [float(ta.data) if hasattr(ta, 'data') else float(ta) for ta in train_accuracy]
  unp_test_acc = [float(va.data) if hasattr(va, 'data') else float(va) for va in test_accuracy]

  # 2. canvas setup
  plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

  # color palette
  COLOR_TRAIN_PRIMARY = "#1A5276"
  COLOR_TEST_SECONDARY = "#E67E22"
  COLOR_GRID = "#E5E8E8"

  # 3. subplot 1 - convergence loss trajectory
  ax1.plot(unp_epochs, unp_train_loss, label="Train", color=COLOR_TRAIN_PRIMARY, linewidth=1.75, alpha=0.95)
  ax1.plot(unp_epochs, unp_test_loss, label="Test / Val", color=COLOR_TEST_SECONDARY, linewidth=1.5, linestyle='--', alpha=0.9)

  ax1.set_title("Optimization Objective (Loss)", fontsize=12, fontweight='bold', pad=12, loc='left')
  ax1.set_xlabel("Optimization Epochs", fontsize=10, labelpad=8)
  ax1.set_ylabel("Mean Squared Error (MSE)", fontsize=10, labelpad=8)

  # structural refinements
  ax1.grid(True, linestyle='-', linewidth=0.5, color=COLOR_GRID)
  ax1.spines[['top', 'right']].set_visible(False)
  ax1.legend(frameon=True, facecolor='white', edgecolor='none', fontsize=9, loc='upper right')

  # 4. subplot 2 - generalization accuracy evolution
  ax2.plot(unp_epochs, unp_train_acc, label="Train", color=COLOR_TRAIN_PRIMARY, linewidth=1.75, alpha=0.95)
  ax2.plot(unp_epochs, unp_test_acc, label="Test / Val", color=COLOR_TEST_SECONDARY, linewidth=1.5, linestyle='--', alpha=0.9)

  # typography and labels
  ax2.set_title("Generalization Capacity (Accuracy)", fontsize=12, fontweight='bold', pad=12, loc='left')
  ax2.set_xlabel("Optimization Epochs", fontsize=10, labelpad=8)
  ax2.set_ylabel("Classification Accuracy (%)", fontsize=10, labelpad=8)
  ax2.set_ylim(-5, 105)

  # global canvas adjustments and save output
  plt.suptitle(f"Model Training Evaluation Report Summary ({model_name.upper()})",
                 fontsize=14, fontweight='semibold', color='#2C3E50', y=0.98)

  plt.tight_layout()

  # export to Workspace
  plt.savefig('metrics_curve.png', bbox_inches='tight', dpi=300)
  print("\n[INFO] Analytics dashboard saved successfully to file 'metrics_curve.png' [DPI=300]")
  plt.show()


def train_model(model, fast_compiled_topology, grid_logits, train_data, train_labels, batch_size, lr):
  '''
  Trains the model in small chunks of size batch_size for full dataset per epoch
  Args:
    model = model used for training
    fast_compiled_topology = static compiled topology generated for speed
    grid_logits = default static logits
    train_data = training dataset
    train_labels = training labels
    batch_size = batch_size per epoch for training model
    lr = learning rate of the model
  Examples:
    train_loss, train_acc = train_model(model=model, fast_compiled_topology=fast_compiled_topology, grid_logits=grid_logits,
                                        train_data=train_dataset, train_labels=train_labels, batch_size=16, lr=0.01)
  '''
  total_epoch_loss = 0.0
  total_correct_preds = 0
  total_samples = 0

  # shuffling the training data
  indices = np.arange(len(train_data))
  np.random.shuffle(indices)
  k = len(train_data) % batch_size
  indices = indices[:-k] if k!=0 else indices

  # training the model
  for i in range(0, len(indices), batch_size):
    batch_indices = indices[i:i+batch_size]
    data_batch = [train_data[index] for index in batch_indices]
    labels_batch = [train_labels[index] for index in batch_indices]

    # forward pass
    fast_compiled_topology.update_batch(data_batch, labels_batch)

    batch_loss = fast_compiled_topology.forward()
    total_epoch_loss = total_epoch_loss + batch_loss

    pred_labels = [1 if logit.data>=0.0 else -1 for logit in grid_logits]

    # backward pass
    for p in model.parameters():
      p.grad = 0.0

    fast_compiled_topology.backward(model.parameters())

    # update weights
    for p in model.parameters():
      p.data = p.data + (-lr/batch_size * p.grad)

    # correct preditions and total samples per batch
    total_correct_preds += sum([pred_label==true_label for pred_label, true_label in zip(pred_labels, labels_batch)])
    total_samples = total_samples + batch_size

  # loss nad accuracy calculations
  total_epoch_loss = total_epoch_loss / total_samples * batch_size
  accuracy = total_correct_preds * 100 / total_samples

  return total_epoch_loss, accuracy


def test_model(model, fast_compiled_topology, grid_logits, test_data, test_labels, batch_size):
  '''
  Tests the model's inference capibilities on test_data divided into chunks of batch_size per epoch
  Args:
    model = model used for training
    fast_compiled_topology = static compiled topology generated for speed
    grid_logits = default static logits
    train_data = training dataset
    train_labels = training labels
    batch_size = batch_size per epoch for training model
    lr = learning rate of the model
  Examples:
    train_loss, train_acc = test_model(model=model, fast_compiled_topology=fast_compiled_topology, grid_logits=grid_logits,
                                        test_data=test_dataset, test_labels=test_labels, batch_size=16)
  '''
  total_epoch_loss = 0.0
  total_correct_preds = 0
  total_samples = 0

  # getting the indices
  indices = np.arange(len(test_data))
  k = len(test_data) % batch_size
  indices = indices[:-k] if k!=0 else indices

  # testing the model
  for i in range(0, len(indices), batch_size):
    batch_indices = indices[i:i+batch_size]
    data_batch = [test_data[index] for index in batch_indices]
    labels_batch = [test_labels[index] for index in batch_indices]

    # forward pass
    fast_compiled_topology.update_batch(data_batch, labels_batch)
    batch_loss = fast_compiled_topology.forward()
    total_epoch_loss = total_epoch_loss + batch_loss

    # total correct predictions and total samples calculation per batch
    pred_labels = [1 if logit.data>=0.0 else -1 for logit in grid_logits]
    total_correct_preds += sum([pred_label==true_label for pred_label, true_label in zip(pred_labels, labels_batch)])
    total_samples = total_samples + batch_size

  # returning total loss and accuracy
  total_epoch_loss = total_epoch_loss / total_samples * batch_size
  accuracy = total_correct_preds * 100 / total_samples
  return total_epoch_loss, accuracy


# Training Custom Binary Classification Model
def main():
  # 1. parser for command line interface
  parser = argparse.ArgumentParser(description="Train an OO-Autograd Multi-Layer Perceptron on a Hyperplane Classification Task.")

  # getting the hyperparameters
  parser.add_argument("--samples", type=int, default=100, help="Total number of samples in the dataset")
  parser.add_argument("--features", type=int, default=3, help="Total number of features per sample")
  parser.add_argument("--low", type=float, default=-2.0, help="Lower bound for per sample value")
  parser.add_argument("--high", type=float, default=2.0, help="Upper bound for per sample value")
  parser.add_argument("--seed", type=int, default=42, help="Random seed anchor for execution reproducibility")
  parser.add_argument("--split", type=float, default=0.8, help="Train/Test dataset partition split ratio")
  parser.add_argument("--epochs", type=int, default=300, help="Number of optimization training epochs")
  parser.add_argument("--lr", type=float, default=0.01, help="Initial learning rate parameter")
  parser.add_argument("--lr_decay", type=float, default=0.99, help="Learning rate decay parameter")

  args = parser.parse_args()

  NUM_SAMPLES = args.samples
  NUM_FEATURES = args.features
  low, high = args.low, args.high
  RANDOM_SEED = args.seed

  SPLIT_VALUE = args.split

  BATCH_SIZE = 16

  epochs = args.epochs
  lr = args.lr
  lr_decay = args.lr_decay
  log_interval = 1 if epochs <= 500 else (epochs//100)

  static_graph_cache = None

  # model metrics storage
  epochs_list = []
  train_loss_list = []
  test_loss_list = []
  train_acc_list = []
  test_acc_list = []

  print(f"Initializing (Samples={NUM_SAMPLES}, Epochs={epochs}, Initial learning rate={lr})")

  # 1. generating the data
  data, labels = generate_data(num_samples=NUM_SAMPLES, num_features=NUM_FEATURES, low=low, high=high, random_seed=RANDOM_SEED)

  # 2. split data into training and testing datasets
  train_data, train_labels, test_data, test_labels = split_data(data=data, labels=labels, split_value=SPLIT_VALUE)

  # memory grid
  memory_grid, target_grid = [], []
  for _ in range(BATCH_SIZE):
    memory_grid.append([Value(0.0) for _ in range(NUM_FEATURES)])
    target_grid.append(Value(0.0))

  # 3. initialize the model
  # model = MLP(NUM_FEATURES, [4, 4, 1])
  model = MLP(NUM_FEATURES, [2, 1])

  # static computation initialization
  grid_logits = [model(x) for x in memory_grid]
  grid_squared_errors = [(yout-ygt)**2 for ygt, yout in zip(target_grid, grid_logits)]

  grid_loss = grid_squared_errors[0]
  for gl in grid_squared_errors[1:]:
    grid_loss = grid_loss + gl

  # initializing prebuilt graph using FastCompiledTopology class
  graph = FastCompiledTopology(loss_node=grid_loss, input_placeholders=memory_grid, target_placeholders=target_grid)

  # training and testing the model
  for epoch in tqdm(range(epochs)):
    # training the model
    train_loss, train_acc = train_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                        train_data=train_data, train_labels=train_labels, batch_size=BATCH_SIZE, lr=lr)

    # testing the model
    test_loss, test_acc = 0.0, 0.0
    with no_grad():
      test_loss, test_acc = test_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                       test_data=test_data, test_labels=test_labels, batch_size=BATCH_SIZE)

    # learning rate adjustments for adaptive learning rate
    lr = lr * lr_decay

    # storing model's metrics data
    if epoch % log_interval == 0 or epochs == epoch-1:
      epochs_list.append(epoch)
      train_loss_list.append(train_loss)
      test_loss_list.append(test_loss)

      train_acc_list.append(train_acc)
      test_acc_list.append(test_acc)

    # display output
    if epoch%10 == 0 or epoch == epochs-1:
      sample_grad = model.parameters()[0].grad
      print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Train Accuracy: {train_acc:.1f}% | Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.1f}%")

  # visualizing the results
  plot_results(model_name="model_0", epochs=epochs_list, train_loss=train_loss_list, test_loss=test_loss_list, train_accuracy=train_acc_list, test_accuracy=test_acc_list)


if __name__ == "__main__":
  main()

In [ ]:
# Testing the model on the xor checkerboard dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from custom_autograd.engine import Value, FastCompiledTopology, no_grad
from custom_autograd.nn import MLP
from custom_autograd.data import generate_xor_checkerboard, split_data
from custom_autograd.train import train_model, test_model


def model_limits():
  NUM_SAMPLES = 200
  NUM_FEATURES = 2
  low, high = -2.0, 2.0
  RANDOM_SEED = 42

  SPLIT_VALUE = 0.8

  BATCH_SIZE = 16

  epochs = 250
  lr = 0.03
  lr_decay = 0.99
  log_interval = 1 if epochs <= 500 else (epochs//100)

  static_graph_cache = None

  epochs_list = []
  train_loss_list = []
  test_loss_list = []
  train_acc_list = []
  test_acc_list = []

  # Generating the data
  X, y = generate_xor_checkerboard(num_samples=NUM_SAMPLES, num_features=NUM_FEATURES, low=low, high=high, random_seed=RANDOM_SEED)

  # Split data into train and test datasets
  train_data, train_labels, test_data, test_labels = split_data(data=X, labels=y, split_value=SPLIT_VALUE)

  # Initializing the model
  # Memory grid
  memory_grid, target_grid = [], []
  for _ in range(BATCH_SIZE):
    memory_grid.append([Value(0.0) for _ in range(NUM_FEATURES)])
    target_grid.append(Value(0.0))

  # 3. Initialize the model
  # model = MLP(NUM_FEATURES, [4, 4, 1])
  model = MLP(NUM_FEATURES, [16, 1])

  # Static computation initialization
  grid_logits = [model(x) for x in memory_grid]
  grid_squared_errors = [(yout-ygt)**2 for ygt, yout in zip(target_grid, grid_logits)]

  grid_loss = grid_squared_errors[0]
  for gl in grid_squared_errors[1:]:
    grid_loss = grid_loss + gl

  graph = FastCompiledTopology(loss_node=grid_loss, input_placeholders=memory_grid, target_placeholders=target_grid)

  # Trainig the model
  for epoch in tqdm(range(epochs)):
    # Training the model
    train_loss, train_acc = train_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                        train_data=train_data, train_labels=train_labels, batch_size=BATCH_SIZE, lr=lr)

    test_loss, test_acc = 0.0, 0.0
    with no_grad():
      test_loss, test_acc = test_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                     test_data=test_data, test_labels=test_labels, batch_size=BATCH_SIZE)

    lr = lr * lr_decay

    if epoch % log_interval == 0 or epochs == epoch-1:
      epochs_list.append(epoch)
      train_loss_list.append(train_loss)
      test_loss_list.append(test_loss)

      train_acc_list.append(train_acc)
      test_acc_list.append(test_acc)


    if epoch%10 == 0 or epoch == epochs-1:
      sample_grad = model.parameters()[0].grad
      print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Train Accuracy: {train_acc:.1f}% | Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.1f}%")

model_limits()

In [ ]:
# Testing model on make_circles dataset

from sklearn.datasets import make_circles

def generate_make_circles(num_samples=1000, noise=0.15):
  X, y = make_circles(n_samples=num_samples, noise=noise, factor=0.4, random_state=42)
  Y = [1.0 if data==1 else -1.0 for data in y]
  return X.tolist(), Y

def model_limits_2():
  NUM_SAMPLES = 200
  NUM_FEATURES = 2
  low, high = -2.0, 2.0
  RANDOM_SEED = 42

  SPLIT_VALUE = 0.8

  BATCH_SIZE = 16

  epochs = 250
  lr = 0.03
  lr_decay = 0.99
  log_interval = 1 if epochs <= 500 else (epochs//100)

  static_graph_cache = None

  epochs_list = []
  train_loss_list = []
  test_loss_list = []
  train_acc_list = []
  test_acc_list = []

  # Generating the data
  X, y = generate_make_circles(num_samples=NUM_SAMPLES, noise=0.15)

  # Split data into train and test datasets
  train_data, train_labels, test_data, test_labels = split_data(data=X, labels=y, split_value=SPLIT_VALUE)

  # Initializing the model
  # Memory grid
  memory_grid, target_grid = [], []
  for _ in range(BATCH_SIZE):
    memory_grid.append([Value(0.0) for _ in range(NUM_FEATURES)])
    target_grid.append(Value(0.0))

  # 3. Initialize the model
  # model = MLP(NUM_FEATURES, [4, 4, 1])
  model = MLP(NUM_FEATURES, [8, 1])

  # Static computation initialization
  grid_logits = [model(x) for x in memory_grid]
  grid_squared_errors = [(yout-ygt)**2 for ygt, yout in zip(target_grid, grid_logits)]

  grid_loss = grid_squared_errors[0]
  for gl in grid_squared_errors[1:]:
    grid_loss = grid_loss + gl

  graph = FastCompiledTopology(loss_node=grid_loss, input_placeholders=memory_grid, target_placeholders=target_grid)

  # Trainig the model
  for epoch in tqdm(range(epochs)):
    # Training the model
    train_loss, train_acc = train_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                        train_data=train_data, train_labels=train_labels, batch_size=BATCH_SIZE, lr=lr)

    test_loss, test_acc = 0.0, 0.0
    with no_grad():
      test_loss, test_acc = test_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                     test_data=test_data, test_labels=test_labels, batch_size=BATCH_SIZE)

    lr = lr * lr_decay

    if epoch % log_interval == 0 or epochs == epoch-1:
      epochs_list.append(epoch)
      train_loss_list.append(train_loss)
      test_loss_list.append(test_loss)

      train_acc_list.append(train_acc)
      test_acc_list.append(test_acc)


    if epoch%10 == 0 or epoch == epochs-1:
      sample_grad = model.parameters()[0].grad
      print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Train Accuracy: {train_acc:.1f}% | Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.1f}%")

model_limits_2()

In [ ]:
# Testing model on twin spirals dataset

import math
import random

def generate_twin_spirals(num_samples=1000, noise=0.05):
  X, y = [], []
  points_per_spiral = num_samples // 2

  for i in range(points_per_spiral):
    # Spiral 1
    theta = (i / points_per_spiral) * 3 * math.pi
    r = (i / points_per_spiral) * 2.0

    x1 = r * math.cos(theta) + random.gauss(0, noise)
    y1 = r * math.sin(theta) + random.gauss(0, noise)
    X.append([x1, y1])
    y.append(1.0)

    # Spiral 2
    x2 = -r * math.cos(theta) + random.gauss(0, noise)
    y2 = -r * math.sin(theta) + random.gauss(0, noise)
    X.append([x2, y2])
    y.append(-1.0)

  indices = np.arange(len(X))
  np.random.shuffle(indices)
  X_shuffled = [X[i] for i in indices]
  y_shuffled = [y[i] for i in indices]

  return X_shuffled, y_shuffled

def model_limits_3():
  NUM_SAMPLES = 200
  NUM_FEATURES = 2
  low, high = -2.0, 2.0
  RANDOM_SEED = 42

  SPLIT_VALUE = 0.8

  BATCH_SIZE = 16

  epochs = 500
  lr = 0.02
  lr_decay = 0.99
  log_interval = 1 if epochs <= 500 else (epochs//100)

  static_graph_cache = None

  epochs_list = []
  train_loss_list = []
  test_loss_list = []
  train_acc_list = []
  test_acc_list = []

  # Generating the data
  X, y = generate_twin_spirals(num_samples=NUM_SAMPLES, noise=0.15)

  # Split data into train and test datasets
  train_data, train_labels, test_data, test_labels = split_data(data=X, labels=y, split_value=SPLIT_VALUE)

  # Initializing the model
  # Memory grid
  memory_grid, target_grid = [], []
  for _ in range(BATCH_SIZE):
    memory_grid.append([Value(0.0) for _ in range(NUM_FEATURES)])
    target_grid.append(Value(0.0))

  # 3. Initialize the model
  # model = MLP(NUM_FEATURES, [4, 4, 1])
  model = MLP(NUM_FEATURES, [32, 32, 1])

  # Static computation initialization
  grid_logits = [model(x) for x in memory_grid]
  grid_squared_errors = [(yout-ygt)**2 for ygt, yout in zip(target_grid, grid_logits)]

  grid_loss = grid_squared_errors[0]
  for gl in grid_squared_errors[1:]:
    grid_loss = grid_loss + gl

  graph = FastCompiledTopology(loss_node=grid_loss, input_placeholders=memory_grid, target_placeholders=target_grid)

  # Trainig the model
  for epoch in tqdm(range(epochs)):
    # Training the model
    train_loss, train_acc = train_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                        train_data=train_data, train_labels=train_labels, batch_size=BATCH_SIZE, lr=lr)

    test_loss, test_acc = 0.0, 0.0
    with no_grad():
      test_loss, test_acc = test_model(model=model, fast_compiled_topology=graph, grid_logits=grid_logits,
                                     test_data=test_data, test_labels=test_labels, batch_size=BATCH_SIZE)

    lr = lr * lr_decay

    if epoch % log_interval == 0 or epochs == epoch-1:
      epochs_list.append(epoch)
      train_loss_list.append(train_loss)
      test_loss_list.append(test_loss)

      train_acc_list.append(train_acc)
      test_acc_list.append(test_acc)


    if epoch%10 == 0 or epoch == epochs-1:
      sample_grad = model.parameters()[0].grad
      print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Train Accuracy: {train_acc:.1f}% | Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.1f}%")

model_limits_3()